# 13. 채널 스냅샷(구독자·총 조회수) 신호 검증
`channel_snapshots`는 08-12 ~ 08-31 하루 2회(구독자, **채널 총 조회수**, 영상 수). 학습 기준일(≤08-05)에 없어 그동안 제외했지만,
**공식 08-25 라벨(666행, 정답 구간 08-25~09-12)** 로 직접 검증할 수 있다: 08-25 시점 스냅샷 피처가 기존 채널 이력 피처 위에 얼마나 더해 주는지 채널 단위 CV로 측정.

In [1]:
import os, glob, platform, warnings
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = {'Windows': 'Malgun Gothic', 'Darwin': 'AppleGothic'}.get(platform.system(), 'NanumGothic')
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 50)

RAW = '../data/raw/epoch_data'
D = '2026-09-01'          # 본선 기준일
FUT = 26                  # 본선 정답 구간 길이 (일)
SEED = 42

con = duckdb.connect()
for p in sorted(glob.glob(f'{RAW}/*.csv')):
    name = os.path.splitext(os.path.basename(p))[0]
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv('{p}', header=true, auto_detect=true)")

# con.sql("SELECT MIN(published_at), MAX(published_at), COUNT(*) FROM video_5d_views").df()

In [2]:
def make_features(Dref):
    f = con.sql(f"""
    WITH lv AS (
        SELECT channel_id, view_5d, like_5d, comment_5d,
               DATE_DIFF('hour', published_at, TIMESTAMP '{Dref}') / 24.0 AS days_ago
        FROM video_5d_views
        WHERE is_shorts = 0 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
    ),
    v5 AS (
        SELECT channel_id,
               COUNT(*)                                        AS n_long_5d,
               MEDIAN(view_5d)                                 AS past_med,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 7)     AS med_7,
               MEDIAN(view_5d) FILTER (WHERE days_ago < 14)    AS med_14,
               MEDIAN(view_5d) FILTER (WHERE days_ago >= 14)   AS med_old,
               MEDIAN(like_5d / NULLIF(view_5d, 0))            AS like_rate,
               MEDIAN(comment_5d / NULLIF(view_5d, 0))         AS comment_rate,
               STDDEV(LN(1 + view_5d))                         AS log_view_std,
               QUANTILE_CONT(LN(1 + view_5d), 0.1)             AS log_q10,
               QUANTILE_CONT(LN(1 + view_5d), 0.9)             AS log_q90,
               REGR_SLOPE(LN(1 + view_5d), -days_ago)          AS trend_slope,
               MIN(days_ago)                                   AS days_since_meas
        FROM lv GROUP BY 1
    ),
    sv AS (
        SELECT channel_id, MEDIAN(view_5d) AS short_med
        FROM video_5d_views
        WHERE is_shorts = 1 AND view_5d IS NOT NULL AND published_at < TIMESTAMP '{Dref}'
        GROUP BY 1
    ),
    up AS (
        SELECT channel_id,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 7 DAY)  AS n_long_7,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_long_14,
               COUNT(*) FILTER (WHERE is_shorts = 0 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 28 DAY) AS n_long_28,
               COUNT(*) FILTER (WHERE is_shorts = 1 AND published_at >= TIMESTAMP '{Dref}' - INTERVAL 14 DAY) AS n_short_14,
               AVG(duration_sec) FILTER (WHERE is_shorts = 0)                                                  AS avg_dur,
               DATE_DIFF('hour', MAX(published_at) FILTER (WHERE is_shorts = 0), TIMESTAMP '{Dref}') / 24.0  AS days_since_long
        FROM videos WHERE published_at < TIMESTAMP '{Dref}' GROUP BY 1
    ),
    games AS (
        SELECT v.channel_id, COUNT(DISTINCT vg.app_id) AS n_games
        FROM videos v LEFT JOIN video_games vg USING (video_id)
        WHERE v.published_at < TIMESTAMP '{Dref}' GROUP BY 1
    )
    SELECT * FROM v5 LEFT JOIN sv USING (channel_id) LEFT JOIN up USING (channel_id)
                     LEFT JOIN games USING (channel_id)
    """).df()

    lp = np.log1p(f.past_med)
    f['log_past_med']      = lp
    f['mom_7']             = np.log1p(f.med_7) - lp
    f['mom_14']            = np.log1p(f.med_14) - lp
    f['mom_old']           = np.log1p(f.med_14) - np.log1p(f.med_old)
    f['short_ratio']       = f.n_short_14 / (f.n_short_14 + f.n_long_14).replace(0, np.nan)
    f['log_short_med_rel'] = np.log1p(f.short_med) - lp
    f['has_game']          = (f.n_games > 0).astype(int)
    f['log_n_long_5d']     = np.log1p(f.n_long_5d)
    return f

FEATURES = ['log_past_med', 'n_long_5d', 'like_rate', 'n_long_14', 'n_short_14', 'short_ratio', 'days_since_long',
            'mom_14', 'log_view_std', 'n_games', 'has_game', 'mom_7', 'mom_old', 'comment_rate',
            'log_q10', 'log_q90', 'trend_slope', 'days_since_meas', 'n_long_7', 'n_long_28', 'avg_dur',
            'log_short_med_rel', 'log_n_long_5d']
print(len(FEATURES), 'features')
make_features(D)[['channel_id'] + FEATURES].head()

23 features


,channel_id,log_past_med,n_long_5d,like_rate,n_long_14,n_short_14,short_ratio,days_since_long,mom_14,log_view_std,n_games,has_game,mom_7,mom_old,comment_rate,log_q10,log_q90,trend_slope,days_since_meas,n_long_7,n_long_28,avg_dur,log_short_med_rel,log_n_long_5d
0,UCM7ccuD_XdAXSTuhVrLUo9A,2.197225,10,0.074423,2,0,0.0000,9.791667,1.170071,1.327668,0,0,NaN,1.170071,0.000000,1.357526,4.121406,-0.011903,9.791667,0,4,1395.846154,NaN,2.397895
1,UCYS2NvZlSjrxZfnn1v7b3QQ,2.079442,35,0.100000,7,0,0.0000,2.333333,0.223144,0.635746,2,1,-0.207639,0.223144,0.000000,1.098612,2.755551,0.012634,2.333333,4,21,23328.500000,NaN,3.583519
2,UCzDyC6JMEXEufozffWzI9Mg,8.734077,9,0.016493,3,13,0.8125,3.916667,0.693389,1.425843,1,1,0.693389,0.765220,0.005982,8.240011,11.572529,0.011885,3.916667,1,5,641.100000,2.462657,2.302585
3,UCBkvEgeJ3_UMwbW7VCxzs8g,6.210600,5,NaN,1,0,0.0000,10.666667,-0.247021,1.102485,0,0,NaN,-1.033102,0.012072,5.993296,8.063446,-0.040266,10.666667,0,1,179.333333,NaN,1.791759
4,UCIsrVj53L57rcLAPOre4Xhg,10.924012,21,0.010228,5,0,0.0000,1.708333,0.203104,0.516451,0,0,0.417460,0.363633,0.001513,10.264408,11.517893,0.010385,1.708333,4,12,1385.185185,NaN,3.091042


In [3]:
def logreg(C=0.1):
    return Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler()),
                     ('m', LogisticRegression(max_iter=2000, C=C, class_weight='balanced'))])

def rank01(x):
    x = pd.Series(np.asarray(x, dtype=float))
    return rankdata(x.fillna(x.median())) / len(x)

def blend(proba, df):
    """모델 확률 순위 + 과거 중앙값 역순위의 평균 (PR-AUC는 순위만 봄)"""
    return (rank01(proba) + rank01(-df.log_past_med)) / 2



## 1. 08-26 급등이 실제 현상인가? — 채널 총 조회수 일일 증가 (view_5d와 독립된 측정)

In [4]:
snap = con.sql("""SELECT channel_id, CAST(collected_at AS DATE) d, MAX(total_view_count) tv, MAX(subscriber_count) sub
                     FROM channel_snapshots WHERE collected_at >= TIMESTAMP '2026-08-12' GROUP BY 1, 2""").df()
snap = snap.sort_values(['channel_id', 'd'])
snap['gain'] = snap.groupby('channel_id').tv.diff()
base = snap[snap.d < pd.Timestamp('2026-08-20')].groupby('channel_id').gain.median()
snap['rel'] = np.log1p(snap.gain.clip(lower=0)) - np.log1p(snap.channel_id.map(base).clip(lower=0))
print(snap.groupby('d').rel.median().round(3).to_string())

d
2026-08-12      NaN
2026-08-13    0.000
2026-08-14    0.000
2026-08-15    0.042
2026-08-16    0.000
2026-08-17    0.000
2026-08-18    0.000
2026-08-19   -0.040
2026-08-20    0.000
2026-08-21   -0.041
2026-08-22    0.000
2026-08-23    0.000
2026-08-24   -0.399
2026-08-25    0.000
2026-08-26    0.000
2026-08-27   -0.035
2026-08-28    0.000
2026-08-29    0.512
2026-08-30    0.480
2026-08-31    0.288


## 2. 08-25 공식 라벨로 스냅샷 피처 검증 (채널 5-fold × 5회 반복)

In [5]:
def snap_feats(Dref, k=7):
    s = con.sql(f"""
      WITH x AS (SELECT channel_id, CAST(collected_at AS DATE) d, MAX(total_view_count) tv, MAX(subscriber_count) sub, MAX(video_count) vc
                 FROM channel_snapshots WHERE collected_at >= TIMESTAMP '2026-08-12' AND collected_at < TIMESTAMP '{Dref}' GROUP BY 1, 2)
      SELECT channel_id,
             ARG_MAX(tv, d) tv_now, ARG_MAX(sub, d) sub_now,
             ARG_MAX(tv, d) - ARG_MAX(tv, d) FILTER (WHERE d <= CAST(TIMESTAMP '{Dref}' - INTERVAL {k} DAY AS DATE)) AS tv_gain_k,
             ARG_MAX(tv, d) FILTER (WHERE d <= CAST(TIMESTAMP '{Dref}' - INTERVAL 3 DAY AS DATE)) AS tv_m3,
             ARG_MAX(sub, d) FILTER (WHERE d <= CAST(TIMESTAMP '{Dref}' - INTERVAL {k} DAY AS DATE)) AS sub_mk
      FROM x GROUP BY 1""").df()
    s['tv_gain_3'] = s.tv_now - s.tv_m3
    out = pd.DataFrame({'channel_id': s.channel_id})
    out['log_tv_day'] = np.log1p(s.tv_gain_k.clip(lower=0) / k)                      # 하루 평균 조회수 증가
    out['tv_accel'] = np.log1p(s.tv_gain_3.clip(lower=0) / 3) - out.log_tv_day        # 최근 3일 가속
    out['sub_growth'] = np.log1p(s.sub_now) - np.log1p(s.sub_mk)
    out['log_sub'] = np.log1p(s.sub_now)
    return out

from sklearn.model_selection import StratifiedKFold
tl = pd.read_csv(f'{RAW}/train_labels.csv')
tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
ev = tl[tl.row_id.str.endswith('2026-08-25')][['channel_id', 'target']]
ev = ev.merge(make_features('2026-08-25'), on='channel_id', how='left').merge(snap_feats('2026-08-25'), on='channel_id', how='left')
ev['tv_rel'] = ev.log_tv_day - ev.log_past_med          # 하루 조회수 / 영상당 조회수 (채널 규모 대비 현재 활동)
SNAP = ['log_tv_day', 'tv_accel', 'sub_growth', 'log_sub', 'tv_rel']
print('스냅샷 결측률:', ev[SNAP].isna().mean().round(3).to_dict())
print('단독 PR-AUC (기저 0.201):', {c: round(average_precision_score(ev.target, ev[c].fillna(ev[c].median())), 3) for c in SNAP})

def cv825(cols, C=0.1, reps=5):
    out = []
    for r in range(reps):
        skf = StratifiedKFold(5, shuffle=True, random_state=r); p = np.zeros(len(ev))
        for a, b in skf.split(ev, ev.target):
            p[b] = logreg(C).fit(ev.iloc[a][cols], ev.target.iloc[a]).predict_proba(ev.iloc[b][cols])[:, 1]
        out.append(average_precision_score(ev.target, p))
    return round(np.mean(out), 4), round(np.std(out), 4)
for name, cols in [('채널 이력 23', FEATURES), ('+ 스냅샷 5', FEATURES + SNAP), ('+ tv_rel·tv_accel', FEATURES + ['tv_rel', 'tv_accel']),
                   ('스냅샷 5만', SNAP + ['log_past_med'])]:
    print(f'{name:16s} PR-AUC', cv825(cols))

스냅샷 결측률: {'log_tv_day': 0.0, 'tv_accel': 0.0, 'sub_growth': 0.0, 'log_sub': 0.0, 'tv_rel': 0.0}
단독 PR-AUC (기저 0.201): {'log_tv_day': 0.141, 'tv_accel': 0.236, 'sub_growth': 0.224, 'log_sub': 0.146, 'tv_rel': 0.276}


채널 이력 23         PR-AUC (np.float64(0.4069), np.float64(0.009))


+ 스냅샷 5          PR-AUC (np.float64(0.4053), np.float64(0.0114))


+ tv_rel·tv_accel PR-AUC (np.float64(0.3987), np.float64(0.0138))
스냅샷 5만           PR-AUC (np.float64(0.3944), np.float64(0.0096))
